# AI-Powered Resume Screening & Candidate Ranking System
### Future Interns Machine Learning Internship — Task 3
**Author:** Machine Learning Intern  
**Date:** August 2026  
**Repository:** FUTURE_ML_O3  

---

## 1. Project Introduction & Business Problem
Recruitment is one of the most critical but resource-intensive processes in any organization. Human Resource (HR) departments frequently receive hundreds or thousands of resumes for a single job opening. Manually screening these documents to filter qualified candidates leads to:
- **High Time-to-Hire:** It takes days or weeks to filter candidates, causing companies to lose high-potential applicants to competitors.
- **Cognitive Fatigue:** Recruiters reading resumes continuously are prone to fatigue, which increases the likelihood of human error.
- **Unconscious Bias:** Subjective screening can lead to inconsistent evaluation metrics.

### Business Solution
This notebook demonstrates a Natural Language Processing (NLP) and Machine Learning prototype to automate candidate screening and ranking. By parsing applicant profiles and matching them against specific job description criteria, the system acts as a transparent, explainable **recruiter decision-support tool** that reduces manual workload.

### Project Objectives
1. Load and clean the Kaggle Resume Dataset (2,400+ entries).
2. Perform NLP text preprocessing and clean resumes while preserving complex technical tags (e.g., C++, .NET, C#).
3. Build a configurable, rule-based technical skill extraction dictionary.
4. Apply TF-IDF Vectorization and Cosine Similarity to compute a textual alignment score between resumes and job requirements.
5. Calculate exact skill match percentages and identify missing skills (skill gap analysis).
6. Combine textual similarity (60% weight) and skill matching (40% weight) to rank candidates and export screening lists.

## 2. Environment Setup & Library Imports
We import core data processing, visualization, machine learning, and natural language libraries.

In [ ]:
import os
import re
import html
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Pre-download NLTK resources
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

# Set plotting aesthetic
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

## 3. Data Loading & Inspection
We load the `Resume.csv` file from our local data folder. We assign anonymous CAND_XXX IDs to candidates to maintain anonymity.

In [ ]:
dataset_path = "../data/Resume.csv"
if not os.path.exists(dataset_path):
    # Fallback to absolute path in case notebook is run from root
    dataset_path = "D:/FUTURE_ML_O3/data/Resume.csv"

df = pd.read_csv(dataset_path)
print("Initial Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

### Dataset Diagnostics
Check for missing values, duplicates, and job category distribution.

In [ ]:
print("Missing Values:")
print(df.isnull().sum())

print("\nDuplicate count in Resume_str:", df.duplicated(subset=["Resume_str"]).sum())

# Clean duplicates and assign CAND_ID
df = df.drop_duplicates(subset=["Resume_str"]).reset_index(drop=True)
df["CAND_ID"] = [f"CAND_{i+1:03d}" for i in range(len(df))]
print("\nPost-deduplication shape:", df.shape)

## 4. Text Preprocessing Pipeline
We build a reusable preprocessing function. Standard tokenizers frequently strip symbols like `+` or `#`, which destroys tags like `C++` and `C#`. We write regex filters to clean HTML, URLs, and punctuation while preserving these terms.

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = html.unescape(text)
    text = text.lower()
    text = re.sub(r'<[^>]*>', ' ', text) # Remove HTML
    text = re.sub(r'http\S+|www\S+', ' ', text) # Remove URLs
    text = re.sub(r'\S+@\S+', ' ', text) # Remove Emails
    text = re.sub(r'\+?\d[\d\-\(\)\s]{8,}\d', ' ', text) # Remove Phones
    text = re.sub(r'[\r\n\t]+', ' ', text)
    # Remove punctuation except selected technical tags (+, #, ., -)
    text = re.sub(r'[^a-zA-Z0-9\s\+\#\.\-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def preprocess_text(text):
    cleaned = clean_text(text)
    if not cleaned:
        return ""
    stop_words = set(stopwords.words('english'))
    if 'c' in stop_words: # Prevent removing C language identifier
        stop_words.remove('c')
        
    # Standardize spaces around technical terms
    cleaned = re.sub(r'(?<=[a-zA-Z])\+', ' +', cleaned)
    cleaned = re.sub(r'\+(?=[a-zA-Z])', '+ ', cleaned)
    
    lemmatizer = WordNetLemmatizer()
    raw_tokens = cleaned.split()
    tokens = []
    for t in raw_tokens:
        if t in ["c++", "c#", ".net"]:
            tokens.append(t)
        elif t not in stop_words and len(t) > 1:
            # Lemmatize
            lemma = lemmatizer.lemmatize(t, pos='v')
            lemma = lemmatizer.lemmatize(lemma, pos='n')
            if lemma.isalpha():
                tokens.append(lemma)
    return " ".join(tokens)

### Apply Preprocessing to Corpus

In [ ]:
print("Preprocessing resumes. This might take a few seconds...")
df["Preprocessed_Resume"] = df["Resume_str"].apply(preprocess_text)
print("Sample preprocessed resume text:")
print(df["Preprocessed_Resume"].iloc[0][:300])

## 5. Supervised Skill Extraction
We create a structured skill dictionary. Each key is the standardized skill name, and the value is a case-insensitive regular expression pattern to detect occurrences.

In [ ]:
SKILL_PATTERNS = {
    "Python": r"\bpython\b",
    "Java": r"\bjava\b(?!script)",
    "C": r"\bc\b",
    "C++": r"\bc\+\+\b",
    "C#": r"\bc\#\b",
    "JavaScript": r"\bjavascript\b|\bjs\b",
    "HTML": r"\bhtml5?\b",
    "CSS": r"\bcss3?\b",
    "SQL": r"\bsql\b",
    "MySQL": r"\bmysql\b",
    "PostgreSQL": r"\bpostgresql\b|\bpostgres\b",
    "MongoDB": r"\bmongodb\b|\bmongo\b",
    "Excel": r"\bexcel\b",
    "Power BI": r"\bpower\s?bi\b",
    "Tableau": r"\btableau\b",
    "Pandas": r"\bpandas\b",
    "NumPy": r"\bnumpy\b",
    "Matplotlib": r"\bmatplotlib\b",
    "Seaborn": r"\bseaborn\b",
    "Scikit-learn": r"\bscikit[-_\s]?learn\b|\bsklearn\b",
    "TensorFlow": r"\btensorflow\b|\btf\b",
    "PyTorch": r"\bpytorch\b",
    "Keras": r"\bkeras\b",
    "Machine Learning": r"\bmachine[-_\s]?learning\b|\bml\b",
    "Deep Learning": r"\bdeep[-_\s]?learning\b|\bdl\b",
    "NLP": r"\bnlp\b|\bnatural\s+language\s+processing\b",
    "Computer Vision": r"\bcomputer\s+vision\b|\bcv\b",
    "Generative AI": r"\bgenerative\s+ai\b|\bgen\s?ai\b",
    "LLM": r"\bllm\b|\blarge\s+language\s+models?\b",
    "RAG": r"\brag\b|\bretrieval\s+augmented\s+generation\b",
    "LangChain": r"\blangchain\b",
    "AWS": r"\baws\b|\bamazon\s+web\s+services\b",
    "Azure": r"\bazure\b",
    "GCP": r"\bgcp\b|\bgoogle\s+cloud\s+(platform\s+)?\b",
    "Docker": r"\bdocker\b",
    "Git": r"\bgit\b|\bgithub\b|\bgitlab\b",
    "Linux": r"\blinux\b|\bunix\b",
    "Django": r"\bdjango\b",
    "Flask": r"\bflask\b",
    "FastAPI": r"\bfastapi\b",
    "Spark": r"\bspark\b|\bpyspark\b|\bapache\s+spark\b",
    "Hadoop": r"\bhadoop\b",
    "Statistics": r"\bstatistics?\b|\bstatistical\b",
    "Data Analysis": r"\bdata\s+analysis\b|\bdata\s+analytics\b",
    "Data Science": r"\bdata\s+science\b",
}

def extract_skills(text):
    if not isinstance(text, str):
        return []
    norm_text = text.lower()
    norm_text = re.sub(r'(?<=[a-zA-Z])\+', ' +', norm_text)
    norm_text = re.sub(r'\+(?=[a-zA-Z])', '+ ', norm_text)
    norm_text = re.sub(r'\s+', ' ', norm_text)
    
    skills = []
    for skill_name, pattern in SKILL_PATTERNS.items():
        if re.search(pattern, norm_text):
            skills.append(skill_name)
    return skills

df["Extracted_Skills"] = df["Resume_str"].apply(extract_skills)
df.head()

## 6. Exploratory Data Analysis & Visualizations
Let's review resume category distribution and top skills.

In [ ]:
# Category Distribution
plt.figure(figsize=(12, 5))
cat_counts = df["Category"].value_counts().head(12)
sns.barplot(x=cat_counts.values, y=cat_counts.index, hue=cat_counts.index, palette="viridis", legend=False)
plt.title("Top 12 Resume Categories in Pool")
plt.xlabel("Count")
plt.show()

# Top Skills
plt.figure(figsize=(10, 5))
all_skills = [s for sublist in df["Extracted_Skills"] for s in sublist]
skills_series = pd.Series(all_skills).value_counts().head(15)
sns.barplot(x=skills_series.values, y=skills_series.index, hue=skills_series.index, palette="crest", legend=False)
plt.title("Top 15 Most Common Skills in Candidate Pool")
plt.xlabel("Frequency")
plt.show()

## 7. Vectorization & Similarity Calculations
We train a `TfidfVectorizer` and fit it on our clean resume corpus.

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), token_pattern=r"(?u)\b\w+\b|\b\w+\+\+\b|\b\w+\#\b")
vectorizer.fit(df["Preprocessed_Resume"])
print("Vectorizer vocabulary size:", len(vectorizer.vocabulary_))

### Define Baseline Job Description
We'll simulate a Machine Learning Engineer job description and screen candidates.

In [ ]:
jd_text = """Looking for a Machine Learning Engineer with Python, SQL, Pandas, NumPy, 
Scikit-learn, Machine Learning, and NLP experience. Experience with AWS, Docker, and Git is preferred."""

# Parse required skills from the JD
required_skills = set(extract_skills(jd_text))
print("Required Skills:", list(required_skills))

# Preprocess JD text
prep_jd = preprocess_text(jd_text)
print("Preprocessed JD:", prep_jd)

### Calculate Cosine Similarity & Skill Gap Analysis

In [ ]:
# Compute Cosine Similarities
resume_tfidf = vectorizer.transform(df["Preprocessed_Resume"])
jd_tfidf = vectorizer.transform([prep_jd])
similarities = cosine_similarity(resume_tfidf, jd_tfidf).flatten() * 100.0

# Skill matching details
skill_matches = []
skill_match_scores = []
matched_skills_list = []
missing_skills_list = []

for idx, row in df.iterrows():
    cand_skills = set(row["Extracted_Skills"])
    matched = cand_skills.intersection(required_skills)
    missing = required_skills.difference(cand_skills)
    
    matched_skills_list.append(", ".join(sorted(list(matched))))
    missing_skills_list.append(", ".join(sorted(list(missing))))
    
    match_pct = (len(matched) / len(required_skills)) * 100.0 if required_skills else 0.0
    skill_match_scores.append(round(match_pct, 2))

df["Similarity_Score"] = np.round(similarities, 2)
df["Skill_Match_Score"] = skill_match_scores
df["Matched_Skills"] = matched_skills_list
df["Missing_Skills"] = missing_skills_list

# Calculate Final score: 60% Similarity + 40% Skill Match
df["Final_Score"] = round(0.60 * df["Similarity_Score"] + 0.40 * df["Skill_Match_Score"], 2)

# Rank
df_ranked = df.sort_values(by="Final_Score", ascending=False).reset_index(drop=True)
df_ranked["Rank"] = df_ranked.index + 1

# Display Top 5 ranked candidates
display_cols = ["Rank", "CAND_ID", "Category", "Similarity_Score", "Skill_Match_Score", "Final_Score", "Matched_Skills"]
df_ranked[display_cols].head()

## 8. Business Insights & Ethical Considerations
### Key Insights
- Candidates with high similarity but low skill match represent applicants with strong resume narrative descriptions but lacking specific requirements (e.g., missing Docker or SQL).
- Candidates with high skill match but lower textual similarity represent developers with raw skills who might not have formatted their descriptions extensively. Recruiters can quickly detect these candidates who would otherwise be overlooked in traditional filters.

### Ethical Guidelines
- **Human in the Loop:** This system serves as a decision-support guide, not a final hiring decision tool. Automatic hiring decisions are strictly prohibited.
- **Keyword Bias:** Candidates who copy-paste keyword terms might rank higher. Hiring teams must review resumes manually to verify context.
- **Sensitive Data Protection:** Resume data must be handled in compliance with privacy regulations (e.g., GDPR, CCPA). Candidate profiles in this project have been anonymized.